In [1]:
import torch
if torch.cuda.is_available():
    print('GPU available: ', torch.cuda.get_device_name(0))

GPU available:  NVIDIA GeForce RTX 2050


solving an example to see How the autograd works....
1. y=x**2
2. y=x**2\
    z=sin(y)
3. y=x**2\
    z=sin(y)
    w=sigma(z)


In [2]:
x=torch.tensor(3., requires_grad=True)
x

tensor(3., requires_grad=True)

In [3]:
y=x**2
y

tensor(9., grad_fn=<PowBackward0>)

In [4]:
y.backward()
x.grad

tensor(6.)

In [5]:
import math

def dz_dx(x):
    return 2*x*math.cos(x**2)

In [6]:
dz_dx(3)

-5.466781571308061

In [7]:
x=torch.tensor(3.0, requires_grad=True)
y=x**2
z=torch.sin(y)

In [8]:
print(x)
print(y)
print(z)


tensor(3., requires_grad=True)
tensor(9., grad_fn=<PowBackward0>)
tensor(0.4121, grad_fn=<SinBackward0>)


In [9]:
## to backtrack or calculating the dz/dx 
z.backward()
x.grad

tensor(-5.4668)

## Example 3 

In [10]:
x=torch.tensor(6.7) ## input Feature
y=torch.tensor(0.0) ## True label

w=torch.tensor(1.0) # weight
b=torch.tensor(0.0) # bias

In [11]:
## Binary cross Entropy loss for scalar
def binary_cross_entropy_loss(target,prediction):
    epsilon = 1e-8   # to prevent log(0)

    prediction = torch.clamp(prediction,epsilon, 1-epsilon)
    return -(target*torch.log(prediction) + (1-target)*torch.log(1-prediction))

In [12]:
## forward pass
z=w*x + b      ## weighted sum of the linear part
y_pred = torch.sigmoid(z)   # predicted Probability

# compute binary cross entropy loss
loss=binary_cross_entropy_loss(y,y_pred)

In [15]:
## Derivatives:
# 1. dL/d(y_pred) : loss with respect to prediction (y_pred)
dloss_dy_pred= (y_pred - y)/(y_pred*(1-y_pred))

# 2. dy_pred/dz : Prediction (y_pred) with respect to z(sigmoid derivative)
dy_pred_by_dz=y_pred*(1-y_pred)

# 3. dz/dw and dz/db: z with respect to w and b
dz_by_dw = x
dz_by_db = 1


##dL_dw=dloss_dy_pred*dy_pred_by_dz*dz_by_dw
dL_dw=(y_pred-y)*x

##dL_db=dloss_dy_pred*dy_pred_by_dz*dz_by_db
dL_db=(y_pred-y)*1

In [16]:
print(f'manual gradient of loss wrt weight(dw): {dL_dw}')
print(f'manual gradient of loss wrt bias(db): {dL_db}')

manual gradient of loss wrt weight(dw): 6.691762447357178
manual gradient of loss wrt bias(db): 0.998770534992218


## By using the Autograd

In [17]:
x=torch.tensor(6.7)
y=torch.tensor(0.0)

In [19]:
w=torch.tensor(1.0, requires_grad=True)
b=torch.tensor(0.0, requires_grad=True)
print(w)
print(b)

tensor(1., requires_grad=True)
tensor(0., requires_grad=True)


In [20]:
z=w*x+b
z

tensor(6.7000, grad_fn=<AddBackward0>)

In [21]:
y_pred=torch.sigmoid(z)
y_pred

tensor(0.9988, grad_fn=<SigmoidBackward0>)

In [22]:
loss=binary_cross_entropy_loss(y,y_pred)
loss

tensor(6.7012, grad_fn=<NegBackward0>)

In [23]:
## for calculating the derivative in the back propagation.....
loss.backward()

In [25]:
# for calculating the gradient for the w and b
print(w.grad,b.grad, sep="\n")

tensor(6.6918)
tensor(0.9988)


previous answer....\
manual gradient of loss wrt weight(dw): 6.691762447357178\
manual gradient of loss wrt bias(db): 0.998770534992218

for the multiple inputs we can also calculate the gradient as....

In [26]:
x=torch.tensor([1.0,2.0,3.0], requires_grad=True)
y=(x**2).mean()
y.backward()
x.grad

tensor([0.6667, 1.3333, 2.0000])

## clearing Gradient

when we run the multiple times then it will add the grading \
it causes problem so we need to clear the gradient.....

In [27]:
x=torch.tensor(2.0, requires_grad=True)

In [ ]:
## run multiple times and see the difference
y=x**2
print(y)
y.backward()
x.grad

tensor(4., grad_fn=<PowBackward0>)


tensor(8.)

In [ ]:
## solution: clearing gradient::: x.grad.zero_()
## now run this multiple times... and you will see no difference in the answer
y=x**2
print(y)
y.backward()
print(x.grad)
x.grad.zero_()

tensor(4., grad_fn=<PowBackward0>)
tensor(4.)


tensor(0.)

## when we need to disable the gradient tracking while prediction 
### it will always requires to save memory and tracking gradient takes memory
example,:
1. we do forward pass with tracking gradient
2. we compute the prediction
3. we do backward pass and calculate the gradient
4. 1. repeat... we do forward without tracking
5. 2. we compute the prediction

In [35]:
## disable Gradient Tracking
x=torch.tensor(2.0, requires_grad=True)
print(x)

tensor(2., requires_grad=True)


In [36]:
y=x**2
y

tensor(4., grad_fn=<PowBackward0>)

In [37]:
y.backward()

In [38]:
x.grad

tensor(4.)

we have 3 options to disable backward tracking:\
1. using requires_grad_(False)
2. detach()
3. torch.no_grad()

In [40]:
#1. using requires_grad_(False)
x.requires_grad_(False)
print(x)
y=x**2
print(y)

tensor(2.)
tensor(4.)


In [ ]:
y.backward()  ## it will not work because we disable the gradient tracking
x.grad

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [ ]:
# 2. detach()
x=torch.tensor(2.0, requires_grad=True)
print(x)
z=x.detach()
y=x**2
print("y-->",y)
y1=z**2
print("y1-->",y1)

# here we can not do y1.backward()

tensor(2., requires_grad=True)
y--> tensor(4., grad_fn=<PowBackward0>)
y1--> tensor(4.)


In [44]:
# 3. torch.no_grad()
x=torch.tensor(2.0, requires_grad=True)
print(x)
with torch.no_grad():
    y=x**2

print(y)

tensor(2., requires_grad=True)
tensor(4.)


In [45]:
y.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn